In [37]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import string

import nltk
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords


from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB,BernoulliNB,ComplementNB,MultinomialNB
from sklearn.neural_network import MLPClassifier



from sklearn.metrics import accuracy_score

In [2]:
np.random.seed(42)

In [3]:
dataset=pd.read_csv('tweet_emotions.csv')

In [4]:
dataset.isna().sum()

tweet_id     0
sentiment    0
content      0
dtype: int64

In [5]:
dataset['sentiment'].value_counts()

sentiment
neutral       8638
worry         8459
happiness     5209
sadness       5165
love          3842
surprise      2187
fun           1776
relief        1526
hate          1323
empty          827
enthusiasm     759
boredom        179
anger          110
Name: count, dtype: int64

In [6]:
labelled_dataset=dataset[dataset['sentiment'] != 'empty']

In [28]:
# extracting 1000 records of each dataset
neutral_dataset = labelled_dataset[labelled_dataset['sentiment'] == 'neutral'][:500]
worry_dataset = labelled_dataset[labelled_dataset['sentiment'] == 'worry'][:500]
happiness_dataset = labelled_dataset[labelled_dataset['sentiment'] == 'happiness'][:500]
sadness_dataset = labelled_dataset[labelled_dataset['sentiment'] == 'sadness'][:500]
love_dataset = labelled_dataset[labelled_dataset['sentiment'] == 'love'][:500]
surprise_dataset = labelled_dataset[labelled_dataset['sentiment'] == 'surprise'][:500]
fun_dataset = labelled_dataset[labelled_dataset['sentiment'] == 'fun'][:500]
relief_dataset = labelled_dataset[labelled_dataset['sentiment'] == 'relief'][:500]
hate_dataset = labelled_dataset[labelled_dataset['sentiment'] == 'hate'][:500]
enthusiasm_dataset = labelled_dataset[labelled_dataset['sentiment'] == 'enthusiasm'][:500]
boredom_dataset = labelled_dataset[labelled_dataset['sentiment'] == 'boredom'][:]
anger_dataset = labelled_dataset[labelled_dataset['sentiment'] == 'anger'][:]

# combining these datasets together
final_training_dataset = pd.concat([neutral_dataset , worry_dataset , happiness_dataset , sadness_dataset , love_dataset , surprise_dataset , 
                                     fun_dataset , relief_dataset , hate_dataset , enthusiasm_dataset , boredom_dataset , anger_dataset],axis = 0)

In [29]:
final_training_dataset

,tweet_id,sentiment,content,processed_corpus
4,1956968416,neutral,@dannycastillo We want to trade with someone w...,"[@, dannycastillo, want, trade, someone, Houst..."
10,1956969456,neutral,cant fall asleep,"[cant, fall, asleep]"
22,1956972116,neutral,No Topic Maps talks at the Balisage Markup Con...,"[Topic, Maps, talk, Balisage, Markup, Conferen..."
31,1956975441,neutral,@cynthia_123 i cant sleep,"[@, cynthia_123, cant, sleep]"
32,1956975860,neutral,I missed the bl***y bus!!!!!!!!,"[missed, bl, *, *, *, bus, !, !, !, !, !, !, !..."
...,...,...,...,...
34762,1752943449,anger,my gawwddd ! 6 headshotss inna row? im on fyaa...,"[gawwddd, !, 6, headshotss, inna, row, ?, im, ..."
35160,1753032343,anger,I'm way to sleepy.. Ill watch my shows lata..G...,"['m, way, sleepy, .., Ill, watch, show, lata, ..."
35913,1753199183,anger,@NerdIndian Take that back. I am insulted.,"[@, NerdIndian, Take, back, ., insulted, .]"
36211,1753257239,anger,@anieszkaa haha i did a ltiitle bit yesterday ...,"[@, anieszkaa, haha, ltiitle, bit, yesterday, ..."


In [48]:
print(string.digits)

0123456789


In [50]:
corpus=[]
lemmatizer=WordNetLemmatizer()
content=final_training_dataset['content']
for document in content:
    tokenized_document = word_tokenize(document)
    filtered_document = [word for word in tokenized_document if word.lower() not in stopwords.words('english') and word.lower() not in string.punctuation and word.lower() not in string.digits] 
    lemmatized_document = [lemmatizer.lemmatize(document) for document in filtered_document]
    corpus.append(lemmatized_document)

In [51]:
final_training_dataset['processed_corpus'] = corpus

In [52]:
final_training_dataset.to_csv('processed_dataset.csv')

In [53]:
corpus = [' '.join(document) for document in corpus]

In [54]:
len(corpus)

5289

In [55]:
final_training_dataset

,tweet_id,sentiment,content,processed_corpus
4,1956968416,neutral,@dannycastillo We want to trade with someone w...,"[dannycastillo, want, trade, someone, Houston,..."
10,1956969456,neutral,cant fall asleep,"[cant, fall, asleep]"
22,1956972116,neutral,No Topic Maps talks at the Balisage Markup Con...,"[Topic, Maps, talk, Balisage, Markup, Conferen..."
31,1956975441,neutral,@cynthia_123 i cant sleep,"[cynthia_123, cant, sleep]"
32,1956975860,neutral,I missed the bl***y bus!!!!!!!!,"[missed, bl, bus]"
...,...,...,...,...
34762,1752943449,anger,my gawwddd ! 6 headshotss inna row? im on fyaa...,"[gawwddd, headshotss, inna, row, im, fyaaahhh]"
35160,1753032343,anger,I'm way to sleepy.. Ill watch my shows lata..G...,"['m, way, sleepy, .., Ill, watch, show, lata, ..."
35913,1753199183,anger,@NerdIndian Take that back. I am insulted.,"[NerdIndian, Take, back, insulted]"
36211,1753257239,anger,@anieszkaa haha i did a ltiitle bit yesterday ...,"[anieszkaa, haha, ltiitle, bit, yesterday, ive..."


In [62]:
vectorizer = TfidfVectorizer()
x = vectorizer.fit_transform(corpus[:])
# x = x.astype('float16')
x = x.toarray()

In [64]:
# GaussianNB,BernoulliNB,ComplementNB,MultinomialNB
y = labelled_dataset['sentiment'][:len(x)]
xtrain,xtest,ytrain,ytest = train_test_split(x, y, test_size=0.2)
gnb=GaussianNB()
mnb=MultinomialNB( force_alpha=True)
mnb.fit(xtrain, ytrain)
gnb.fit(xtrain, ytrain)
ypred_gnb = gnb.predict(xtest)
ypred_mnb = mnb.predict(xtest)
accuracy_gnb=accuracy_score(ytest,ypred_gnb)
accuracy_mnb=accuracy_score(ytest,ypred_mnb)
print(accuracy_gnb)
print(accuracy_mnb)

0.17485822306238186
0.31947069943289225


In [65]:
import pickle
pickle.dump(gnb , open('trained_model.pkl','wb'))
pickle.dump(vectorizer,open('vectorizer.pkl','wb'))